# Overview

In this notebook, we automate what we have created in the manual deployment workflow and use a Databricks jobs to run each stage automatically. However, we've added a manual approval stage so that a model that is deployed to production must first be approved my the manager or else the job will fail.


In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs

# Automated Deployment Workflow with Manual Approval

This creates an automated deployment workflow with manual approval:

### Staging Deployment -> Validation -> Manual Approval -> Production Deployment
* Deploy to staging
* Run validation tests
* Display approval gate (manual review required)
* Deploys to production only after human approval


In [0]:
model_name = "ml_catalog.titanic_schema.example_titanic_model"
model_version = "3"

In [0]:
job_name = "Titanic_Staging_Auto_Deployment_Workflow"

user = "millikanevan@gmail.com"
job_config = {
    "name": job_name,
    "tasks": [
        # Task 1: Deploy to Staging
        jobs.Task(
            task_key="deploy_to_staging",
            description="Deploy model to staging endpoint",
            notebook_task=jobs.NotebookTask(
                notebook_path=f"/Repos/{user}/enablement_lab/deployment_jobs/deploy_to_staging",
                base_parameters={
                    "model_version": "{{job.parameters.model_version}}",
                    "endpoint_name": "example-titanic-serving-stg"
                }
            ),
            max_retries=0 
        ),
        
        # Task 2: Run Validation Tests
        jobs.Task(
            task_key="run_validation_tests",
            description="Run load tests and validation on staging endpoint",
            depends_on=[jobs.TaskDependency(task_key="deploy_to_staging")],
            notebook_task=jobs.NotebookTask(
                notebook_path=f"/Repos/{user}/enablement_lab/deployment_jobs/run_tests",
                base_parameters={
                    "endpoint_name": "example-titanic-serving-stg",
                    "num_requests": "5000",
                    "num_workers": "10", # Change this to 10 so it can pass
                    "p99_threshold_ms": "250"
                }
            ),
            max_retries=0
        ),
        
        # Task 3: Approval Gate (Manual Review)
        jobs.Task(
            task_key="approval_gate",
            description="Display test results and wait for manual approval",
            depends_on=[jobs.TaskDependency(task_key="run_validation_tests")],
            notebook_task=jobs.NotebookTask(
                notebook_path=f"/Repos/{user}/enablement_lab/deployment_jobs/approval",
                base_parameters={
                    "model_name" : "{{job.parameters.model_name}}",
                    "model_version": "{{job.parameters.model_version}}",
                    "approval_tag_name" : "{{task.name}}" # By default, set the approval_tag_name as the task name. And it will fail until a reviewer manually change it to "Approved"
                }
            ),
            max_retries=0
        ),

        # Task 4: Deployment to Production
        jobs.Task(
            task_key="deploy_to_production",
            description="Deploy model to production endpoint",
            depends_on=[jobs.TaskDependency(task_key="approval_gate")],
            notebook_task=jobs.NotebookTask(
                notebook_path=f"/Repos/{user}/enablement_lab/deployment_jobs/deploy_to_production",
                base_parameters={
                    "model_version": "{{job.parameters.model_version}}",
                    "endpoint_name": "example-titanic-serving-prd"
                }
            ),
            max_retries=0
        )
    ],

    "parameters": [
        jobs.JobParameter(name="model_name", default=model_name),
        jobs.JobParameter(name="model_version", default=model_version)
    ],
    
    "max_concurrent_runs": 1
}

In [0]:
# Running this cell will create the job

w = WorkspaceClient()
try:
    created_job = w.jobs.create(**job_config)
    job_id = created_job.job_id
    print(f"Deployment Job created successfully!")
    print(f"Job ID: {job_id}")
    print(f"Job Name: {job_name}")
    print(f"View job at: https://{w.config.host}/jobs/{job_id}")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Job '{job_name}' already exists.")
        existing_jobs = w.jobs.list(name=job_name)
        for job in existing_jobs:
            job_id = job.job_id
            print(f"Existing Job 1 ID: {job_id}")
    else:
        print(f"Error creating job 1: {e}")
        raise

print("\n" + "="*60 + "\n")


After creating the job, update the registered model with the deployment job

In [0]:
import mlflow
from mlflow.tracking.client import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")  
client.update_registered_model(model_name, deployment_job_id=created_job.job_id)


Then, trigger the deployment job

In [0]:
# Trigger Job
model_version_to_deploy = "3" # You can choose which model version you want to deploy

try:
    run = w.jobs.run_now(
        job_id=job_id,
        job_parameters={"model_version": model_version_to_deploy}
    )
    print(f"Deployment Job Triggered")
    print(f"Run ID: {run.run_id}")
    print(f"\nMonitor run at: https://{w.config.host}/jobs/{job_id}/runs/{run.run_id}")
except Exception as e:
    print(f"Error triggering deployment job: {e}")